# Treino de especialista — Kaggle

Configuração do notebook (painel direito):

- **Accelerator**: GPU T4 x2 (ou P100)
- **Internet**: ligado — é necessário para clonar o repositório
- **Persistence**: Files only

Mantenha o notebook **privado**. Nunca envie `.env`, chaves de API ou tokens: o treino usa apenas dados históricos públicos.

A sessão do Kaggle cai após 12 h (9 h sem interação). O treino salva checkpoints em `/kaggle/working`, que sobrevive à queda da sessão.

In [ ]:
# ---------------------------------------------------------------- CONFIG
AGENT = "bear"        # bull | bear | ranger
TIMESTEPS = 300_000
# -------------------------------------------------------------------------
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "SEM GPU — ative em Settings > Accelerator")

In [ ]:
%cd /kaggle/working
!rm -rf repo
!git clone --depth 1 https://github.com/drtassio/BinanceFuturesTrader.git repo
%cd /kaggle/working/repo
!pip install -q -r cloud/requirements-cloud.txt

In [ ]:
import subprocess, sys
for script in ('scripts/build_causal_dataset.py', 'scripts/build_meta_features.py'):
    result = subprocess.run([sys.executable, script])
    assert result.returncode == 0, f'{script} falhou'


In [ ]:
# Se esta celula falhar, NAO treine: alguma feature ainda enxerga o futuro.
import subprocess, sys
assert subprocess.run([sys.executable, 'scripts/verify_causality.py']).returncode == 0, 'vazamento detectado'


In [ ]:
import subprocess, sys
cmd = [sys.executable, "cloud/train_agent.py", "--agent", AGENT, "--timesteps", str(TIMESTEPS)]
print(" ".join(cmd))
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
print("exit code:", process.wait())
assert process.returncode == 0, 'treino falhou'


In [ ]:
import json, pathlib
report = json.loads(pathlib.Path(f"cloud/artifacts/{AGENT}_training_report.json").read_text())
verdict = report["verdict"]
print("APROVADO PARA OPERAR:", verdict["approved_for_live_trading"])
for name, passed in verdict["checks"].items():
    print(f"  [{'OK ' if passed else 'NAO'}] {name}")
print()
print(f"trades no holdout : {verdict['holdout_trades']}")
print(f"retorno liquido   : {verdict['holdout_net_return']*100:+.2f}%")
print(f"buy & hold        : {verdict['buy_and_hold_return']*100:+.2f}%")
print(f"drawdown maximo   : {verdict['holdout_max_drawdown']*100:.1f}%")

In [ ]:
# Empacota a pasta da execucao INTEIRA em /kaggle/working (aba Output).
# Politica, scaler e contrato de features precisam viajar juntos.
import pathlib, tarfile
runs = sorted(p for p in pathlib.Path('cloud/artifacts').glob(f'{AGENT}_2*')
              if (p / 'models' / f'{AGENT}_specialist_sac.zip').exists())
assert runs, 'nenhuma execucao com modelo salvo'
run = runs[-1]
archive = f'/kaggle/working/{AGENT}_artifacts.tar.gz'
with tarfile.open(archive, 'w:gz') as tar:
    for item in run.iterdir():
        if item.name != 'checkpoints':
            tar.add(item, arcname=item.name)
    tar.add(f'cloud/artifacts/{AGENT}_training_report.json', arcname=f'{AGENT}_training_report.json')
print('pacote:', archive)
print('no seu PC: python scripts/promote_model.py --agent', AGENT, '--archive <arquivo baixado>')
